# Notebook 2: Data Preparation & Embedding Generation

This notebook loads the raw CSV files, performs cleaning and standardisation, and saves cleaned versions.  
It also generates OpenAI embeddings for the cleaned review texts and the PXI questionnaire.

**Operations**:
- Remove duplicates based on `reviewId` (if exists)
- Drop rows with empty content
- Convert `at` to datetime, extract year/month/day
- Rename columns for consistency (e.g., `thumbsUpCount` -> `likes`)
- Keep only necessary columns: `reviewId`, `userName`, `content`, `score`, `likes`, `at`
- Generate embeddings for cleaned reviews (CoC and RoK) using OpenAI API
- Generate embeddings for PXI questionnaire items

**Outputs**:
- `dataset/coc_clean.csv`
- `dataset/rok_clean.csv`
- `processed/coc_embeddings_*.pkl` (multiple files, merged automatically)
- `processed/rok_embeddings_*.pkl`
- `processed/pxi_embeddings.pkl`

**Note**: Generating embeddings requires a valid OpenAI API key in a `.env` file and may take up to an hour for 30k+ reviews. The processed pkl files are used directly in Notebook 3 to avoid re-running API calls.

Run all cells to generate cleaned data and embeddings.

In [ ]:
# cleaning CoC and RoK reviews
import pandas as pd
import os

# Configuration
INPUT_DIR = "dataset"
OUTPUT_DIR = "dataset"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def clean_reviews(input_file, output_file):
    # Load raw data
    df = pd.read_csv(input_file)
    
    # Remove duplicates if reviewId exists
    if 'reviewId' in df.columns:
        df = df.drop_duplicates(subset=['reviewId'])
    
    # Drop rows without content
    if 'content' in df.columns:
        df = df.dropna(subset=['content'])
        df = df[df['content'].str.strip() != '']
        df['content'] = df['content'].str.replace('\n', ' ').str.replace('\r', ' ').str.strip()
        df = df[df['content'].str.len() >= 5]
    
    # Convert timestamp
    if 'at' in df.columns:
        df['at'] = pd.to_datetime(df['at'], errors='coerce')
        df = df.dropna(subset=['at'])
        df['year'] = df['at'].dt.year
        df['month'] = df['at'].dt.month
        df['date'] = df['at'].dt.date
    
    # Rename thumbsUpCount to likes
    if 'thumbsUpCount' in df.columns:
        df = df.rename(columns={'thumbsUpCount': 'likes'})
    
    # Convert score and likes to numeric
    if 'score' in df.columns:
        df['score'] = pd.to_numeric(df['score'], errors='coerce').fillna(0)
    if 'likes' in df.columns:
        df['likes'] = pd.to_numeric(df['likes'], errors='coerce').fillna(0)
    
    # Keep only relevant columns
    keep_cols = ['reviewId', 'userName', 'content', 'score', 'likes', 'at', 'year', 'month', 'date']
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols]
    
    # Save cleaned data
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"Saved {len(df)} rows to {output_file}")
    return df

# Clean Clash of Clans
coc_raw = os.path.join(INPUT_DIR, "clashofclans.csv")
coc_clean = os.path.join(OUTPUT_DIR, "coc_clean.csv")
if os.path.exists(coc_raw):
    clean_reviews(coc_raw, coc_clean)
else:
    print(f"File not found: {coc_raw}")

# Clean Rise of Kingdoms
rok_raw = os.path.join(INPUT_DIR, "riseofkingdoms.csv")
rok_clean = os.path.join(OUTPUT_DIR, "rok_clean.csv")
if os.path.exists(rok_raw):
    clean_reviews(rok_raw, rok_clean)
else:
    print(f"File not found: {rok_raw}")

Saved 29701 rows to dataset/coc_clean.csv
Saved 28337 rows to dataset/rok_clean.csv


In [ ]:
# reviews embeddings
os.makedirs("processed", exist_ok=True)

# Generate embeddings for cleaned reviews (run once, may take time)
from utils import get_embeddings

get_embeddings('dataset/coc_clean.csv', 'processed/coc_embeddings', 'review', text_col='content')
get_embeddings('dataset/rok_clean.csv', 'processed/rok_embeddings', 'review', text_col='content')

Saved 500 rows to processed/rok_embeddings_500.pkl
Saved 500 rows to processed/rok_embeddings_1000.pkl
Saved 500 rows to processed/rok_embeddings_1500.pkl
Saved 500 rows to processed/rok_embeddings_2000.pkl
Saved 500 rows to processed/rok_embeddings_2500.pkl
Saved 500 rows to processed/rok_embeddings_3000.pkl
Saved 500 rows to processed/rok_embeddings_3500.pkl
Saved 500 rows to processed/rok_embeddings_4000.pkl
Saved 500 rows to processed/rok_embeddings_4500.pkl
Saved 500 rows to processed/rok_embeddings_5000.pkl
Saved 500 rows to processed/rok_embeddings_5500.pkl
Saved 500 rows to processed/rok_embeddings_6000.pkl
Saved 500 rows to processed/rok_embeddings_6500.pkl
Saved 500 rows to processed/rok_embeddings_7000.pkl
Saved 500 rows to processed/rok_embeddings_7500.pkl
Saved 500 rows to processed/rok_embeddings_8000.pkl
Saved 500 rows to processed/rok_embeddings_8500.pkl
Saved 500 rows to processed/rok_embeddings_9000.pkl
Saved 500 rows to processed/rok_embeddings_9500.pkl
Saved 500 row

In [ ]:
import pickle
import glob

def merge_embeddings(prefix, output_path):
    """
    Merge multiple pkl files with prefix (e.g., 'processed/coc_embeddings_')
    into a single DataFrame and save as output_path.
    """
    files = glob.glob(f"{prefix}_*.pkl")
    if not files:
        print(f"No files found for {prefix}")
        return
    dfs = []
    for f in files:
        with open(f, 'rb') as fp:
            dfs.append(pd.DataFrame(pickle.load(fp)))
    merged = pd.concat(dfs, ignore_index=True)
    with open(output_path, 'wb') as fp:
        pickle.dump(merged.to_dict(), fp)
    print(f"Merged {len(files)} files into {output_path}, total rows: {len(merged)}")

# Merge CoC embeddings
merge_embeddings('processed/coc_embeddings', 'processed/coc_embeddings_full.pkl')

# Merge RoK embeddings
merge_embeddings('processed/rok_embeddings', 'processed/rok_embeddings_full.pkl')

Merged 60 files into processed/coc_embeddings_full.pkl, total rows: 29700
Merged 57 files into processed/rok_embeddings_full.pkl, total rows: 28336


In [4]:
# Generate PXI questionnaire embeddings (only once)
import pandas as pd
# Assuming pxi_queries.csv is in dataset/
get_embeddings('dataset/pxi_queries.csv', 'processed/pxi_embeddings', 'query')

Saved 30 rows to processed/pxi_embeddings.pkl
Embedding generation complete.
